In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt -n 3

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [4]:
import collections
import numpy as np
import tqdm

readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
requests = list()
docembs = collections.defaultdict(dict)

with open("stand/log.local.txt") as f:
    req = list()
    reqid = None
    models = list()
    prevreqmodel = None
    reqmodel = dict()
    prevmodelid = -1
    bannermodelid = -1
    for i, line in tqdm.tqdm_notebook(enumerate(f)):
        if line.startswith("Model = 6;"):
            prevreqmodel = reqmodel
            reqmodel = dict()
        
        if line.startswith("dbid 184394"):
            print(line),
            print(prevmodelid)
            
        if line.startswith("Model = "):
            spl = line.split(" ")
            prevmodelid = int(spl[2][:-1])
            bannermodelid = max(bannermodelid , prevmodelid)
            reqmodel[prevmodelid] = readvector(spl[3])
        elif line.startswith("dbid"):
            spl = line.split(" ")
            dbid = int(spl[1][:-1])
            docembs[bannermodelid][dbid] = readvector(spl[2])
        elif line.startswith("seed"):
            if len(requests) >= 1000:
                break
            if req:
                requests.append((reqid, prevreqmodel, sorted(req)))
                req = list()
            reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
        else:
            req.append(
                (int(line.split()[0]), float(line.split()[1]))
            )

requests = [r for r in requests if len(r[2]) == 16514]

dbid 184394; [-0.0512872,0.0193015,0.113894,0.0252779,-0.155716,0.123881,-0.0206214,0.0627056,-0.0278315,0.372551,0.000806784,-0.0653532,-0.0687121,0.0878991,0.0476131,0.140274,-0.170451,0.0202663,0.141594,-0.066827,0.0701703,-0.130011,-0.0651865,0.0546216,0.152157,0.229269,-0.126042,0.0172359,-0.0209199,0.0127505,-0.122559,-0.0714047,0.062702,0.0793581,0.0914689,0.207112,0.179825,0.0887367,-0.168106,0.0249156,-0.119067,-0.0720207,-0.0911074,-0.0435966,0.028035,0.0111506,0.0389887,-0.112137,-0.0404294,-0.0934756,0.0164074,-0.0955409,0.00808596,-0.0411514,-0.0539788,0.00297383,0.0199476,0.134285,0.0632445,-0.0619632,-0.110289,0.0151936,0.135167,0.105939,-0.079014,-0.0997647,0.000153098,-0.0108563,0.0231189,0.0272427,-0.0255719,-0.0711633,-0.0625287,0.0397355,0.0102844,-0.149684,0.0828743,0.142888,-0.0176144,-0.0116712,0.0371071,-0.212492,-0.0232992,-0.0258352,-0.0230406,-0.0762529,-0.0378617,-0.041919,-0.091265,-0.0543666,-0.0468707,0.00201314,0.128076,-0.284469,0.0124104,0.0697693,0.05

In [6]:
[(i, len(docembs[i].keys())) for i in docembs]  # should be equal

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]

In [7]:
docblocks = {
    mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
    for mid in docembs
}
docblocks

{6: array([[ 0.0407761 , -0.0866397 ,  0.0973855 , ..., -0.0337863 ,
          0.1449    ,  0.023338  ],
        [-0.0173765 ,  0.103833  ,  0.0541691 , ...,  0.0947965 ,
          0.119532  ,  0.111093  ],
        [-0.010643  , -0.0188779 ,  0.128833  , ..., -0.0185986 ,
          0.0270978 ,  0.0370004 ],
        ...,
        [-0.0822528 ,  0.0734386 ,  0.0820323 , ..., -0.0283938 ,
          0.0727261 ,  0.244681  ],
        [-0.0396752 ,  0.215461  ,  0.148155  , ..., -0.0594413 ,
          0.0757048 ,  0.140465  ],
        [-0.0272712 ,  0.207217  ,  0.154168  , ..., -0.00856583,
          0.0459061 ,  0.18192   ]]),
 7: array([[-0.0457229 ,  0.0874625 ,  0.0102194 , ...,  0.0135331 ,
         -0.0444551 ,  0.012255  ],
        [-0.135033  ,  0.0949462 , -0.0646434 , ...,  0.105708  ,
         -0.0441772 ,  0.0313092 ],
        [-0.176407  ,  0.15634   , -0.0423812 , ...,  0.165932  ,
         -0.0977578 , -0.0507538 ],
        ...,
        [ 0.0745855 ,  0.15481   , -0.0633103 , 

In [9]:
(requests[0][1][6] ** 2).sum()  # normalized embed

0.9994653378993866

In [206]:
class EvalContext:
    def __init__(self, games_count = 16514, key_size = 100, train_size = 0.7):
        self.games_count = games_count
        self.key_games = np.random.choice(np.arange(games_count), key_size, replace=False)
        
        self.requests = requests
        np.random.shuffle(self.requests)
        
        self.key_reqs = self.requests[:key_size + 1]
        self.key_reqs_idx = np.arange(key_size + 1)
        
        self.train_split = int(len(self.requests) * train_size)
        
        assert key_size + 1 < self.train_split
        
        self.train_reqs = self.requests[key_size + 1: self.train_split]
        self.test_reqs = self.requests[self.train_split:]
        
        self.slices = ["key", "train", "test"]
        print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))
        
        self.docblocks = docblocks
        
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [207]:
ctx = EvalContext()

101 589 297


In [208]:
ctx.get_requests("test")[0][1]

{6: array([-0.262534  , -0.114519  ,  0.00502034,  0.0614758 , -0.0348453 ,
         0.0329322 , -0.00308267,  0.145801  , -0.0174197 ,  0.0669361 ,
        -0.122934  ,  0.147582  ,  0.201714  , -0.278539  , -0.111861  ,
        -0.170204  ,  0.117949  , -0.0203089 , -0.0141248 , -0.0314362 ,
         0.0711987 ,  0.0823294 ,  0.0625961 , -0.0886118 ,  0.0302026 ,
         0.102934  , -0.0620531 ,  0.0489378 , -0.00403892, -0.00916403,
        -0.0281767 ,  0.107867  ,  0.0331392 , -0.222312  ,  0.129077  ,
         0.0622441 , -0.0270416 , -0.0129953 ,  0.0519121 ,  0.0405255 ,
        -0.0338513 ,  0.0366273 ,  0.0892287 , -0.0794731 , -0.0615317 ,
        -0.0683076 ,  0.193883  ,  0.0137861 , -0.144305  , -0.104568  ,
        -0.0623583 , -0.102486  , -0.182329  , -0.0504768 ,  0.0240938 ,
        -0.056903  , -0.0405863 ,  0.00697353, -0.0238853 , -0.0198067 ,
        -0.187711  ,  0.0500037 , -0.0417955 , -0.110733  ,  0.155068  ,
         0.0766266 , -0.10485   , -0.0185744 , -

In [209]:
ctx.get_requests("test")[0][2]

[(96239, 0.00820086617),
 (96240, 0.00283994549),
 (96242, 0.008155154064),
 (96243, 0.0107728662),
 (96245, 13.98484993),
 (96246, 0.001738305436),
 (96247, 0.04069347307),
 (96248, 0.009355150163),
 (96249, 0.01000765804),
 (96250, 0.0820203051),
 (96252, 0.06509402394),
 (96253, 0.001938444679),
 (96254, 0.01832463779),
 (96255, 0.007787247654),
 (96256, 0.04530210793),
 (96258, 0.009699990973),
 (96261, 0.009750030003),
 (96262, 0.02202042378),
 (96263, 0.05060986057),
 (96264, 0.007727455813),
 (96265, 0.01011274755),
 (96266, 0.04072244838),
 (96269, 0.02497417852),
 (96271, 0.001246328815),
 (96272, 0.003124461975),
 (96275, 0.006716758478),
 (96276, 0.01911381632),
 (96278, 0.04537538439),
 (96279, 0.01825784706),
 (96282, 0.003672034014),
 (96283, 0.007812587544),
 (96285, 0.01264383085),
 (96287, 0.01575783081),
 (96289, 0.00805203896),
 (96290, 0.02026026323),
 (96291, 0.03857922554),
 (96292, 0.0619716458),
 (96293, 0.00582418032),
 (96294, 0.002944943029),
 (96295, 0.00330

In [277]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in self.ctx.get_requests(t)
        ])
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top = np.argsort(-self.game_avg_scores[t])

    def recommend(self, t):
        return [self.top for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):
        results = list()
        for rec, tru in zip(self.recommend(t), self.get_user_scores(t)):
            tru = np.argsort(-tru)
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            results.append(ev(rec[:topsize], tru[:topsize]) / float(topsize))
        print(np.mean(results), len(results))
        return np.mean(results)

In [278]:
p = Popular(ctx)
p.fit()
p.get_score("test")

0.4622558922558923 297


0.4622558922558923

In [309]:
import tensorflow as tf

class CBKnnV0(Popular):
    def __init__(self, ctx):
        super().__init__(ctx)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[2][i][1] for i in ctx.key_games])
                    for r_i in ctx.get_requests(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[2][g_i][1] for r in ctx.get_requests("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in ctx.get_requests("key")
        ])
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return self.embed_users[t]
    
    def get_game_embs(self):
        return self.embed_games

    def fit(self, learning_rate = 0.001, n = 500, c = 5000, train_popbias = False, verbose=True):
        self.train_popbias = train_popbias

        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        
        if verbose:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        loss = tf.keras.losses.MeanSquaredError()
        opt =  tf.keras.optimizers.Adam(learning_rate=learning_rate)

        for i in range(n):
            def loss():
                logits = train_user_embs @ self.W @ game_embs.T

                if self.train_popbias:
                    logits += self.pb * self.game_avg_scores["train"]

                v = (
                    tf.reduce_mean((train_user_scores - logits) ** 2)
                    + tf.reduce_mean(self.W * self.W) * c
                )
                
                if verbose:
                    print(v.numpy())
                
                return v

            opt.minimize(loss, [self.W, self.pb] if train_popbias else [self.W]).numpy()

    def recommend(self, t):
        logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]
            
        return np.argsort(-logits, axis=1)
    
    # inherited
    # def get_score(self, t, topsize = 100):

In [269]:
ck = CBKnnV0(ctx)

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)


In [270]:
ck.fit(c = 1000)

self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-6.0956227e-04,  1.7034950e-03,  7.1513443e-04, ...,
         3.8146065e-04, -5.2730888e-05,  6.9700403e-04],
       [ 1.0280013e-05,  7.9647213e-04, -1.5913417e-03, ...,
         1.6639202e-03,  1.7180342e-03,  1.4339763e-03],
       [ 2.8468072e-04,  6.8913086e-04,  1.5367210e-03, ...,
         1.4826459e-03,  5.4771797e-04, -1.6131952e-04],
       ...,
       [ 9.4179210e-05,  6.9712388e-04, -1.5946325e-03, ...,
        -1.3938948e-04, -9.8494743e-04,  1.1264175e-03],
       [ 1.1844394e-03, -2.5582538e-04, -2.1245137e-04, ...,
         1.4228194e-03, -1.0175538e-03, -1.6055773e-03],
       [-1.0212491e-03,  7.6899352e-04, -5.2385352e-04, ...,
        -1.6996589e-03,  5.5879058e-04, -2.3117214e-05]], dtype=float32)>
0-loss =  tf.Tensor(1.5740757665707659, shape=(), dtype=float64)
1.5864408
1.570859
1.5398903
1.5199076
1.5246351
1.5279264
1.5181296
1.5056633
1.5006081
1.5022081
1.5044326
1.5032086
1.49

In [271]:
ck.get_score("test")

0.5775420875420875 297


0.5775420875420875

In [300]:
ck = CBKnnV0(ctx)
ck.fit(c = 1000, train_popbias=True)

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.1932388e-03, -2.4563074e-06, -1.6932657e-03, ...,
        -9.6632069e-04,  1.7034077e-03,  7.6523930e-04],
       [-1.4139570e-03,  1.4125338e-03, -7.2554336e-04, ...,
         2.7357191e-05,  1.6172763e-03,  4.1495636e-04],
       [ 6.5413298e-04, -6.3910306e-04, -2.2276670e-04, ...,
         4.2099357e-04,  4.7692581e-04,  7.4985891e-04],
       ...,
       [ 2.7859941e-04,  2.2165329e-04,  4.8311130e-04, ...,
         5.5581809e-04,  6.4936699e-04,  1.4477110e-03],
       [-3.8029253e-04, -1.6782809e-03, -9.2544005e-04, ...,
        -1.5322581e-03, -7.1597024e-04, -5.3095887e-04],
       [ 8.6514413e-04, -1.3474494e-03,  8.4124773e-04, ...,
         1.2763330e-

In [301]:
ck.get_score("test")

0.5584511784511785 297


0.5584511784511785

In [302]:
ck.pb

<tf.Variable 'Variable:0' shape=() dtype=float32, numpy=0.7116165>

In [310]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid):
        super().__init__(ctx)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games

    """def fit(self, learning_rate = 0.001, n = 500, c = 5000):
        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        print("self.W = ", self.W)
        
        print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        loss = tf.keras.losses.MeanSquaredError()
        opt =  tf.keras.optimizers.Adam(learning_rate=learning_rate)

        for i in range(n):
            def loss():
                v = (
                    tf.reduce_mean((train_user_scores - train_user_embs @ self.W @ game_embs.T) ** 2)
                    + tf.reduce_mean(self.W * self.W) * c
                )
                print(v.numpy())
                return v

            opt.minimize(loss, [self.W]).numpy()

    def recommend(self, t):
        logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T
        return np.argsort(-logits, axis=1)"""

In [314]:
d = DssmKnn(ctx, 6)
d.fit(n = 1, c = 1, train_popbias=True, verbose=False)

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)


In [327]:
d.W = np.eye(*d.W.shape) * 5
d.pb = 1.
d.get_score("test")

0.5865993265993265 297


0.5865993265993265

# нужны ранжирующие лоссы!

In [328]:
score = dict()
for c in [0, .1, 1, 10, 100, 1000]:
    for m in [6,7,8,9]:
        d = DssmKnn(ctx, m)
        d.fit(c = c, train_popbias=True, verbose=False)
        score_ = d.get_score("test")
        print("c, m = ", c, m, " score = ", score_)
        score[(c, m)] = score_

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5259259259259259 297
c, m =  0 6  score =  0.5259259259259259
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5195622895622896 297
c, m =  0 7  score =  0.5195622895622896
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5234680134680134 297
c, m =  0 8  score =  0.5234680134680134
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 10

In [329]:
score

{(0, 6): 0.5259259259259259,
 (0, 7): 0.5195622895622896,
 (0, 8): 0.5234680134680134,
 (0, 9): 0.4782154882154882,
 (0.1, 6): 0.522828282828283,
 (0.1, 7): 0.5148821548821549,
 (0.1, 8): 0.5221548821548821,
 (0.1, 9): 0.48114478114478115,
 (1, 6): 0.5064309764309766,
 (1, 7): 0.498013468013468,
 (1, 8): 0.5055218855218855,
 (1, 9): 0.48134680134680136,
 (10, 6): 0.48468013468013466,
 (10, 7): 0.4780808080808081,
 (10, 8): 0.4825252525252525,
 (10, 9): 0.47090909090909094,
 (100, 6): 0.46986531986531993,
 (100, 7): 0.46508417508417504,
 (100, 8): 0.46791245791245784,
 (100, 9): 0.46430976430976434,
 (1000, 6): 0.4622895622895623,
 (1000, 7): 0.46232323232323236,
 (1000, 8): 0.4622558922558923,
 (1000, 9): 0.4622558922558923}

In [331]:
for c in [0, .1, 1, 10, 100, 1000]:
    d = CBKnnV0(ctx)
    d.fit(c = c, train_popbias=True, verbose=False)
    score_ = d.get_score("test")
    print("c, m = ", c, m, " score = ", score_)

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5901346801346801 297
c, m =  0 9  score =  0.5901346801346801
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5897643097643097 297
c, m =  0.1 9  score =  0.5897643097643097
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
0.5896632996632997 297
c, m =  1 9  score =  0.5896632996632997
self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 

In [306]:
d = DssmKnn(ctx, 6)
d.fit(c = 1, train_popbias=True)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-1.4697642e-03, -2.7478769e-04,  7.7716562e-05, ...,
        -2.4155855e-04,  1.4623210e-03,  2.9899820e-04],
       [-6.1691145e-04, -1.4614398e-03,  1.4817846e-03, ...,
        -1.0085157e-03,  1.2587246e-03, -1.3093433e-03],
       [-2.1342785e-04,  1.1311808e-03, -8.3102845e-04, ...,
         2.8979927e-04,  6.3440338e-04,  1.4369604e-03],
       ...,
       [-9.2752068e-04, -9.0627099e-04,  1.3497210e-03, ...,
         1.6692501e-03, -1.7112714e-03,  5.4743217e-04],
       [ 7.4095726e-05,  9.1107725e-04, -1.3292815e-03, ...,
         1.9994646e-04,  8.6200028e-04, -3.5686002e-04],
       [ 6.0386508e-05, -1.5753498e-03, -1.0803365e-03, ...,
         1.6614151e-

0.5064309764309766

In [307]:
d = DssmKnn(ctx, 6)
d.fit(c = 100, train_popbias=True)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 1.2492409e-03, -1.4148499e-03,  1.7120210e-03, ...,
        -8.2630379e-04, -1.6309388e-03, -1.3440347e-03],
       [ 8.3598314e-04,  1.1619225e-03,  7.6468376e-04, ...,
        -9.5402653e-04, -2.0092398e-04, -1.4697809e-03],
       [ 1.4214456e-03, -1.1981652e-03,  8.0036552e-04, ...,
        -5.0705351e-04, -8.3587400e-04,  1.1114990e-03],
       ...,
       [ 1.0042653e-03, -1.2597977e-03, -1.5414911e-03, ...,
         1.2204326e-03,  1.4168172e-04, -6.8137096e-04],
       [-1.6852686e-03,  9.2339667e-04, -1.5701278e-03, ...,
        -1.1018030e-03,  1.4371299e-03, -2.3741872e-05],
       [-3.8884133e-05, -6.2101823e-04,  2.9980348e-04, ...,
        -1.3238954e-

0.4698316498316498

In [254]:
d = DssmKnn(ctx, 7)
d.fit(c = 1)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-6.11129435e-05, -1.76232308e-04,  1.41824785e-05, ...,
         3.65616666e-04, -8.56306578e-04, -1.49085035e-03],
       [-1.47908356e-03, -2.86363967e-04, -6.16044272e-04, ...,
        -1.32064905e-03,  2.41133268e-04,  1.66671013e-03],
       [ 1.48483808e-03, -1.61870121e-04,  7.82507355e-04, ...,
         1.60615053e-03, -1.56653370e-03, -1.51129672e-03],
       ...,
       [ 1.73844543e-04, -1.01559411e-03,  4.64571873e-04, ...,
         5.49237127e-04,  7.01189037e-06, -5.15758176e-04],
       [-2.21498165e-04,  2.90570257e-04, -8.06919881e-04, ...,
        -9.88341853e-05,  2.25062366e-04,  4.36053262e-04],
       [ 1.05637044e-03, -9.90804518e-04, -9.875099

0.3786531986531987

In [255]:
d = DssmKnn(ctx, 8)
d.fit(c = 1)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-4.4546247e-04,  2.9320328e-04,  2.4530693e-04, ...,
        -3.2944532e-05,  1.1207670e-04,  5.9416889e-05],
       [ 1.1628222e-03,  7.3469477e-04,  4.5831845e-04, ...,
        -1.4984932e-03, -6.9928193e-04,  1.2813688e-03],
       [-1.0526223e-03,  9.3454568e-05,  5.4018794e-05, ...,
        -4.4130237e-04,  3.4278454e-04, -6.1394973e-04],
       ...,
       [ 8.0066681e-04,  9.9898432e-04, -1.3502075e-03, ...,
         1.3873848e-03, -9.8830822e-04,  9.6929254e-04],
       [-1.1236859e-03, -1.2905857e-03,  9.6759736e-04, ...,
         1.5442723e-03, -1.7683566e-04,  1.5473274e-03],
       [-1.7254748e-03,  1.6248965e-03,  6.2829553e-05, ...,
        -1.3067016e-

0.4675084175084176

In [256]:
d = DssmKnn(ctx, 9)
d.fit()
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 0.00112605,  0.0003794 ,  0.00083465, ...,  0.00102953,
        -0.00117538,  0.00017341],
       [ 0.00092484, -0.00076796, -0.0002561 , ..., -0.00071062,
         0.00166632, -0.00097806],
       [ 0.00086191, -0.00087855,  0.00063438, ..., -0.00029437,
         0.00064299, -0.00083264],
       ...,
       [ 0.0006576 ,  0.00066541, -0.00066413, ..., -0.00098837,
        -0.00064531,  0.00137999],
       [ 0.00161982,  0.00087691,  0.00080164, ..., -0.00012547,
        -0.00112681,  0.00064834],
       [ 0.00105329, -0.00131144,  0.00013109, ...,  0.00039437,
        -0.00030353,  0.00119232]], dtype=float32)>
0-loss =  tf.Tensor(1.5740757665707659, shape=(), dtyp

0.003636363636363637

In [257]:
d = DssmKnn(ctx, 9)
d.fit(c=1)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[ 5.2413868e-04, -3.8008450e-04,  9.6854835e-04, ...,
         1.3692534e-03, -3.7117704e-04, -1.5385922e-03],
       [ 9.5627515e-04,  1.5304971e-03,  3.2107189e-04, ...,
        -6.0649717e-04, -1.8585697e-04, -1.4334568e-03],
       [-8.8575459e-04, -1.5695913e-03,  3.2784059e-04, ...,
         1.0600134e-03,  2.2593140e-05,  7.8529568e-04],
       ...,
       [ 9.3838602e-04, -4.5177102e-04, -1.4454131e-03, ...,
         1.0579941e-03, -6.0885801e-04,  8.1680831e-04],
       [-4.7238692e-04,  8.6211681e-04, -4.1873977e-04, ...,
        -1.1349391e-03,  1.5710351e-03,  1.1693505e-03],
       [ 1.6254875e-03, -3.8906009e-04,  1.1138878e-03, ...,
         1.5459699e-

0.2929629629629629

In [258]:
d = DssmKnn(ctx, 9)
d.fit(c=10)
d.get_score("test")

self.embed_users['train'].shape =  (589, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (589, 101)
self.W =  <tf.Variable 'Variable:0' shape=(100, 100) dtype=float32, numpy=
array([[-1.1128330e-03,  7.9970300e-04,  1.5583363e-03, ...,
         3.7382782e-04, -4.3210297e-04,  1.0967058e-03],
       [ 1.0122559e-03, -9.2477084e-04, -8.5562392e-04, ...,
        -1.5380479e-03,  1.3975659e-03,  7.4283854e-04],
       [ 7.9818099e-04,  1.2382376e-03, -1.6428074e-03, ...,
        -2.4159282e-04, -6.2837993e-04, -1.6822751e-03],
       ...,
       [-1.5067407e-03,  1.7158571e-03,  8.2942846e-05, ...,
         7.2645099e-04, -5.4503576e-04, -1.5780389e-04],
       [ 6.4738735e-04,  4.3454766e-06,  1.7000618e-03, ...,
        -5.6545297e-04,  1.2226286e-03,  1.0671434e-03],
       [-3.0594080e-04,  4.8015534e-04, -7.5513631e-04, ...,
         1.6388803e-

0.28114478114478114

# EOLN

In [127]:
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from sklearn.metrics import pairwise 

In [ ]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

In [23]:
import matplotlib.pyplot as plt
plt.hist(embed_games.flatten())

(array([1.667877e+06, 2.000000e+01, 6.000000e+00, 4.000000e+00,
        1.000000e+00, 4.000000e+00, 0.000000e+00, 0.000000e+00,
        1.000000e+00, 1.000000e+00]),
 array([9.74706709e-05, 6.56539696e+01, 1.31307842e+02, 1.96961714e+02,
        2.62615586e+02, 3.28269458e+02, 3.93923330e+02, 4.59577202e+02,
        5.25231074e+02, 5.90884946e+02, 6.56538818e+02]),
 <a list of 10 Patch objects>)

In [25]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.005857787810383747 0.4473702031602709
d -> 0.4814559819413093 0.006038374717832957 0.4473702031602709
c -> 0.01746049661399549 0.005959367945823928 0.4473702031602709
w -> 0.4944808126410835 0.0060835214446952595 0.4473702031602709


In [26]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.003182844243792325 0.3336343115124153
d -> 0.36880361173814896 0.0031828442437923255 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.38158013544018066 0.003069977426636569 0.3336343115124153


Тюним

In [135]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 1000
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)
1.3766173
1.2021382
1.1933919
1.1919646
1.1917517
1.1917268
1.1917244
1.1917244
1.1917244
1.1917243


In [28]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.002618510158013544 0.3336343115124153
d -> 0.36880361173814896 0.0029119638826185104 0.3336343115124153
c -> 0.008577878103837472 0.002957110609480813 0.3336343115124153
w -> 0.40388261851015805 0.0027539503386004517 0.3336343115124153


In [29]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.006422121896162527 0.4473702031602709
d -> 0.4814559819413093 0.0060609480812641075 0.4473702031602709
c -> 0.01746049661399549 0.005711060948081265 0.4473702031602709
w -> 0.512020316027088 0.005970654627539504 0.4473702031602709


# dssm?

In [136]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-docblocks[i] @ requests_i[1][i] for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.15483069977426636 0.0031602708803611735 0.3336343115124153
7 -> 0.11663656884875846 0.0034085778781038373 0.3336343115124153
8 -> 0.15386004514672688 0.0032054176072234763 0.3336343115124153
9 -> 0.0362979683972912 0.0032054176072234763 0.3336343115124153
e -> 0.3762076749435666 0.0031602708803611735 0.3336343115124153
d -> 0.36880361173814896 0.0028442437923250565 0.3336343115124153
c -> 0.008577878103837472 0.0029119638826185104 0.3336343115124153
w -> 0.40388261851015805 0.002618510158013544 0.3336343115124153


In [137]:
games_top

array([0.0047713 , 0.00415357, 0.00380251, ..., 0.1423562 , 0.04750573,
       0.07614477])

In [141]:
dssmid = 7
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
7 -> 0.3336343115124153 0.0031602708803611735 0.3336343115124153
x =  1
7 -> 0.35379232505643343 0.0027765237020316025 0.3336343115124153
x =  2
7 -> 0.37090293453724604 0.0031602708803611735 0.3336343115124153
x =  3
7 -> 0.38564334085778784 0.002889390519187359 0.3336343115124153
x =  4
7 -> 0.40259593679458244 0.002663656884875847 0.3336343115124153
x =  5
7 -> 0.4182844243792325 0.002641083521444696 0.3336343115124153
x =  6
7 -> 0.4323702031602709 0.0031602708803611743 0.3336343115124153
x =  7
7 -> 0.4396839729119639 0.0028668171557562076 0.3336343115124153
x =  8
7 -> 0.44162528216704294 0.0034085778781038373 0.3336343115124153
x =  9
7 -> 0.4411286681715576 0.0035665914221218965 0.3336343115124153
x =  10
7 -> 0.4392099322799097 0.0030248306997742664 0.3336343115124153
x =  11
7 -> 0.43462753950338606 0.0030248306997742664 0.3336343115124153
x =  12
7 -> 0.4294356659142212 0.003386004514672686 0.3336343115124153
x =  13
7 -> 0.42151241534988715 0.003363431151241535 0.333

In [142]:
dssmid = 6
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.3336343115124153 0.0030925507900677203 0.3336343115124153
x =  1
6 -> 0.37266365688487585 0.003002257336343115 0.3336343115124153
x =  2
6 -> 0.3972911963882618 0.003340857787810384 0.3336343115124153
x =  3
6 -> 0.42458239277652365 0.0024830699774266367 0.3336343115124153
x =  4
6 -> 0.45449209932279916 0.003115124153498871 0.3336343115124153
x =  5
6 -> 0.4766591422121897 0.0030699774266365687 0.3336343115124153
x =  6
6 -> 0.4895033860045147 0.0032054176072234763 0.3336343115124153
x =  7
6 -> 0.4941083521444696 0.0032731376975169302 0.3336343115124153
x =  8
6 -> 0.49146726862302487 0.003002257336343115 0.3336343115124153
x =  9
6 -> 0.48632054176072237 0.002595936794582393 0.3336343115124153
x =  10
6 -> 0.4764559819413092 0.0028893905191873593 0.3336343115124153
x =  11
6 -> 0.4654853273137698 0.0027313769751693 0.3336343115124153
x =  12
6 -> 0.45309255079006777 0.0030248306997742672 0.3336343115124153
x =  13
6 -> 0.43932279909706545 0.003024830699774266 0.3336343

In [152]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.3336343115124153 0.0029345372460496616 0.3336343115124153
x =  1
9 -> 0.34738148984198647 0.002844243792325057 0.3336343115124153
x =  2
9 -> 0.3593227990970655 0.003002257336343115 0.3336343115124153
x =  3
9 -> 0.3699097065462754 0.0030248306997742672 0.3336343115124153
x =  4
9 -> 0.37024830699774264 0.003024830699774266 0.3336343115124153
x =  5
9 -> 0.3647178329571106 0.002595936794582393 0.3336343115124153
x =  6
9 -> 0.3520767494356659 0.0028668171557562076 0.3336343115124153
x =  7
9 -> 0.3291196388261851 0.003047404063205418 0.3336343115124153
x =  8
9 -> 0.30002257336343113 0.002641083521444696 0.3336343115124153
x =  9
9 -> 0.2689390519187359 0.002889390519187359 0.3336343115124153
x =  10
9 -> 0.24376975169300227 0.0027313769751693 0.3336343115124153
x =  11
9 -> 0.22112866817155757 0.0031828442437923255 0.3336343115124153
x =  12
9 -> 0.20067720090293456 0.0034085778781038373 0.3336343115124153
x =  13
9 -> 0.18354401805869075 0.0030925507900677203 0.33363431

In [143]:
for x in range(20):
    print("x = ", x)
    rank_w = np.argsort(-x*embed_users @ W @ embed_games.T-games_top, axis=1)


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_w, "w"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
w -> 0.3336343115124153 0.0032279909706546274 0.3336343115124153
x =  1
w -> 0.39647855530474035 0.0033182844243792326 0.3336343115124153
x =  2
w -> 0.40907449209932284 0.0030925507900677203 0.3336343115124153
x =  3
w -> 0.4110835214446953 0.0031828442437923255 0.3336343115124153
x =  4
w -> 0.41049661399548537 0.0035214446952595937 0.3336343115124153
x =  5
w -> 0.4100451467268624 0.0034085778781038373 0.3336343115124153
x =  6
w -> 0.4096839729119639 0.003340857787810384 0.3336343115124153
x =  7
w -> 0.40936794582392777 0.0034762979683972913 0.3336343115124153
x =  8
w -> 0.40880361173814905 0.0031602708803611735 0.3336343115124153
x =  9
w -> 0.4087810383747179 0.0030248306997742664 0.3336343115124153
x =  10
w -> 0.4089164785553048 0.002934537246049662 0.3336343115124153
x =  11
w -> 0.4089164785553048 0.0027313769751693 0.3336343115124153
x =  12
w -> 0.40887133182844243 0.003295711060948081 0.3336343115124153
x =  13
w -> 0.4086004514672686 0.002595936794582393 0.333634

In [147]:
print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)


In [148]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T-games_top) ** 2)
            + tf.reduce_mean(W * W) * 150
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
1.3189981
1.1485099
1.1218199
1.110669
1.1049802
1.1020124
1.1004989
1.0997444
1.0993747
1.0991968


In [149]:
rank_w = np.argsort(-embed_users @ W @ embed_games.T-games_top, axis=1)


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_w, "w"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids

            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

w -> 0.4295936794582393 0.002595936794582393 0.3336343115124153


In [41]:
500 -> .4108
250 -> .4117
150 -> .4143
100 -> .4151
050 -> .4142
010 -> .4115
000 -> .4033

SyntaxError: invalid syntax (<ipython-input-41-1b4e77c0b5a7>, line 1)

In [150]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.4941083521444696 0.0028216704288939053 0.3336343115124153
7 -> 0.4396839729119639 0.0032054176072234763 0.3336343115124153
8 -> 0.47376975169300223 0.0033182844243792326 0.3336343115124153
9 -> 0.3291196388261851 0.0028668171557562076 0.3336343115124153
e -> 0.3762076749435666 0.0031376975169300227 0.3336343115124153
d -> 0.36880361173814896 0.0027539503386004517 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.4295936794582393 0.0032054176072234763 0.3336343115124153


In [153]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5568735891647856 0.006151241534988712 0.4473702031602709
7 -> 0.5192212189616252 0.005846501128668171 0.4473702031602709
8 -> 0.567178329571106 0.006455981941309256 0.4473702031602709
9 -> 0.37682844243792324 0.006264108352144469 0.4473702031602709
e -> 0.4731151241534989 0.006218961625282167 0.4473702031602709
d -> 0.4814559819413093 0.006060948081264108 0.4473702031602709
c -> 0.01746049661399549 0.005812641083521445 0.4473702031602709
w -> 0.527155756207675 0.006060948081264108 0.4473702031602709


In [154]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-5 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5759367945823928 0.005711060948081263 0.4473702031602709
7 -> 0.5242776523702032 0.006015801354401806 0.4473702031602709
8 -> 0.5645372460496614 0.005744920993227991 0.4473702031602709
9 -> 0.44339729119638827 0.006015801354401806 0.4473702031602709
e -> 0.4731151241534989 0.005970654627539504 0.4473702031602709
d -> 0.4814559819413093 0.006049661399548532 0.4473702031602709
c -> 0.01746049661399549 0.006309255079006772 0.4473702031602709
w -> 0.527155756207675 0.006230248306997742 0.4473702031602709


In [160]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 200
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.666410835214447 0.01167607223476298 0.5744469525959367
7 -> 0.6234255079006772 0.011834085778781037 0.5744469525959367
8 -> 0.6523137697516931 0.012178329571106095 0.5744469525959367
9 -> 0.5699266365688487 0.011811512415349886 0.5744469525959367
e -> 0.5722686230248306 0.01195823927765237 0.5744469525959367
d -> 0.5961343115124152 0.012076749435665914 0.5744469525959367
c -> 0.030299097065462757 0.012753950338600452 0.5744469525959367
w -> 0.626089164785553 0.011726862302483071 0.5744469525959367


In [161]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 250
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.6831467268623025 0.014785553047404065 0.5979503386004515
7 -> 0.6498690744920992 0.015480812641083523 0.5979503386004515
8 -> 0.6795124153498872 0.01450112866817156 0.5979503386004515
9 -> 0.5858916478555304 0.014961625282167042 0.5979503386004515
e -> 0.592334085778781 0.015431151241534989 0.5979503386004515
d -> 0.6212821670428893 0.015327313769751695 0.5979503386004515
c -> 0.03572460496613995 0.015449209932279913 0.5979503386004515
w -> 0.6479593679458239 0.015462753950338602 0.5979503386004515


In [51]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.5744469525959367 0.012121896162528218 0.5744469525959367
x =  1
6 -> 0.6088205417607224 0.01252257336343115 0.5744469525959367
x =  2
6 -> 0.6479627539503386 0.01204288939051919 0.5744469525959367
x =  3
6 -> 0.664283295711061 0.012088036117381492 0.5744469525959367
x =  4
6 -> 0.6479740406320542 0.011580135440180587 0.5744469525959367
x =  5
6 -> 0.6167832957110609 0.011839729119638827 0.5744469525959367
x =  6
6 -> 0.5822968397291196 0.012025959367945822 0.5744469525959367
x =  7
6 -> 0.5485609480812641 0.012291196388261852 0.5744469525959367
x =  8
6 -> 0.5183069977426636 0.012127539503386006 0.5744469525959367
x =  9
6 -> 0.49157449209932275 0.011930022573363432 0.5744469525959367
x =  10
6 -> 0.46870203160270885 0.012020316027088036 0.5744469525959367
x =  11
6 -> 0.44742099322799095 0.012477426636568848 0.5744469525959367
x =  12
6 -> 0.4283465011286682 0.012116252821670429 0.5744469525959367
x =  13
6 -> 0.41200338600451464 0.012003386004514673 0.5744469525959367
x

In [52]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.4473702031602709 0.0059480812641083515 0.4473702031602709
x =  1
6 -> 0.4830135440180587 0.006241534988713319 0.4473702031602709
x =  2
6 -> 0.5132167042889391 0.006478555304740407 0.4473702031602709
x =  3
6 -> 0.5453160270880361 0.006038374717832957 0.4473702031602709
x =  4
6 -> 0.56813769751693 0.006207674943566591 0.4473702031602709
x =  5
6 -> 0.5727878103837472 0.006015801354401806 0.4473702031602709
x =  6
6 -> 0.5652934537246049 0.005857787810383747 0.4473702031602709
x =  7
6 -> 0.5506546275395033 0.005575620767494356 0.4473702031602709
x =  8
6 -> 0.5308352144469526 0.005744920993227991 0.4473702031602709
x =  9
6 -> 0.5122234762979685 0.006106094808126411 0.4473702031602709
x =  10
6 -> 0.4936230248306998 0.006591422121896162 0.4473702031602709
x =  11
6 -> 0.47469525959367953 0.00618510158013544 0.4473702031602709
x =  12
6 -> 0.45573363431151237 0.006218961625282167 0.4473702031602709
x =  13
6 -> 0.43795711060948084 0.0059480812641083515 0.4473702031602709


In [158]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.4473702031602709 0.0063318284424379225 0.4473702031602709
x =  1
9 -> 0.45654627539503384 0.005846501128668171 0.4473702031602709
x =  2
9 -> 0.4679345372460497 0.006331828442437923 0.4473702031602709
x =  3
9 -> 0.4737810383747178 0.0060270880361173815 0.4473702031602709
x =  4
9 -> 0.46619638826185095 0.006218961625282167 0.4473702031602709
x =  5
9 -> 0.44339729119638827 0.006399548532731377 0.4473702031602709
x =  6
9 -> 0.4103386004514672 0.006399548532731377 0.4473702031602709
x =  7
9 -> 0.37682844243792324 0.005688487584650113 0.4473702031602709
x =  8
9 -> 0.3405079006772009 0.005654627539503386 0.4473702031602709
x =  9
9 -> 0.30753950338600455 0.0060270880361173815 0.4473702031602709
x =  10
9 -> 0.27742663656884875 0.006309255079006772 0.4473702031602709
x =  11
9 -> 0.25196388261851016 0.006162528216704289 0.4473702031602709
x =  12
9 -> 0.23016930022573365 0.005948081264108352 0.4473702031602709
x =  13
9 -> 0.2112641083521445 0.006252821670428894 0.44737020

In [159]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.5744469525959367 0.012392776523702033 0.5744469525959367
x =  1
9 -> 0.584018058690745 0.012020316027088036 0.5744469525959367
x =  2
9 -> 0.586410835214447 0.011653498871331828 0.5744469525959367
x =  3
9 -> 0.5699266365688487 0.012279909706546277 0.5744469525959367
x =  4
9 -> 0.5346388261851016 0.011822799097065465 0.5744469525959367
x =  5
9 -> 0.49005079006772007 0.012133182844243792 0.5744469525959367
x =  6
9 -> 0.44170993227990973 0.011918735891647854 0.5744469525959367
x =  7
9 -> 0.39646726862302484 0.012466139954853276 0.5744469525959367
x =  8
9 -> 0.35639954853273137 0.011952595936794583 0.5744469525959367
x =  9
9 -> 0.32224040632054174 0.011997742663656885 0.5744469525959367
x =  10
9 -> 0.2926354401805869 0.012042889390519187 0.5744469525959367
x =  11
9 -> 0.2685214446952596 0.01260722347629797 0.5744469525959367
x =  12
9 -> 0.2483803611738149 0.012641083521444697 0.5744469525959367
x =  13
9 -> 0.23149548532731376 0.012127539503386006 0.5744469525959367